# MgO Defect Thermodynamics from Scratch

## Milestone 1: Host structure and defect setup

### Research Question

How does the Fermi level affect the formation energy and charge-state
stability of a defect in MgO?

### Physics

For a given defect charge state q, the formation energy depends linearly
on the Fermi level:

E_f(D^q, E_F) = C_q + qE_F

where C_q contains the terms that are independent of the Fermi level.

### Goal

1. Calculate formation energies across the band gap.
2. Plot formation-energy lines for different charge states.
3. Determine the thermodynamically stable charge state.
4. Identify charge transition levels.

## Module 1.1 Transition PORCAR into dictionary

In [44]:
from pathlib import Path
import numpy as np
import json

In [ ]:
# find the path to the POSCAR file
notebook_dir = Path.cwd()
project_root = notebook_dir.parent

data_root = (
    project_root
    / "data"
    / "raw"
    / "mgo_defect_thermodynamics_dataset"
)

poscar_path = (
    data_root
    / "02_bulk_relaxed"
    / "POSCAR"
)

print("Current directory:", Path.cwd())
print("POSCAR path:", poscar_path)
print("File exists:", poscar_path.exists())

Current directory: /Users/yu/Desktop/defect_dynamic/notebooks
POSCAR path: /Users/yu/Desktop/defect_dynamic/data/raw/mgo_defect_thermodynamics_dataset/02_bulk_relaxed/POSCAR
File exists: True


In [11]:
# Load the POSCAR file

text = poscar_path.read_text(encoding="utf-8")
lines = text.splitlines()

print(type(text))
print(type(lines))
print(len(lines))
print(lines[:8])

<class 'str'>
<class 'list'>
72
['MgO 64-atom initial supercell', '1.0', '8.424000 0.000000 0.000000', '0.000000 8.424000 0.000000', '0.000000 0.000000 8.424000', 'Mg O', '32 32', 'Direct']


In [14]:
# Parse the title and scaling factor
title = lines[0].strip()
scale = float(lines[1].strip())

print("title:", title)
print("scale:", scale)
print(type(title))
print(type(scale))

title: MgO 64-atom initial supercell
scale: 1.0
<class 'str'>
<class 'float'>


In [ ]:
# Parse the lattice vectors
lattice_rows = lines[2:5]
lattice_data = []

for row in lattice_rows:
    row_data = [float(x) for x in row.split()]
    lattice_data.append(row_data)

lattice = np.array(lattice_data) * scale

print(lattice)
print(lattice.shape)



[[8.424 0.    0.   ]
 [0.    8.424 0.   ]
 [0.    0.    8.424]]
(3, 3)


In [ ]:
# Parse elements and their counts
elements = (lines[5].split()) #get a list of elements from the 6th line of the POSCAR file
counts = [int(value) for value in lines[6].split()]
natoms = sum(int(count) for count in counts)

print("elements:", elements)
print("counts:", counts)
print("natoms:", natoms)

assert elements == ["Mg", "O"]
assert counts == [32, 32]
assert natoms == 64

elements: ['Mg', 'O']
counts: [32, 32]
natoms: 64


In [20]:
#Parse coordinate mode
coordinate_mode = lines[7].strip()

print("coordinate mode:", coordinate_mode)
assert coordinate_mode.lower() == "direct"

coordinate mode: Direct


In [ ]:
# Calculate volume of the unit cell
volume_A3 = np.linalg.det(lattice)
volume = np.abs(volume_A3) 
print("Volume:", volume, "Å³")

Volume: 597.7988490240002 Å³


In [25]:
# Parse into a dictionary
header = {
    "title": title,
    "scale": scale,
    "lattice": lattice,
    "elements": elements,
    "counts": counts,
    "natoms": natoms,
    "coordinate_mode": coordinate_mode,
    "volume": volume
}

for key, value in header.items():
    print(key, ":", value)

title : MgO 64-atom initial supercell
scale : 1.0
lattice : [[8.424 0.    0.   ]
 [0.    8.424 0.   ]
 [0.    0.    8.424]]
elements : ['Mg', 'O']
counts : [32, 32]
natoms : 64
coordinate_mode : Direct
volume : 597.7988490240002


In [40]:
# Function
def parse_poscar_header(path: Path) -> dict:
    # TODO：拆分行
    text = path.read_text(encoding="utf-8")
    lines = text.splitlines()

    # TODO：解析 title 和 scale
    title = lines[0].strip()
    scale = float(lines[1].strip())

    # TODO：解析 lattice
    lattice_rows = lines[2:5]
    lattice_data = []
    for row in lattice_rows:
        row_data = [float(x) for x in row.split()]
        lattice_data.append(row_data)
    lattice = np.array(lattice_data) * scale

    # TODO：解析 elements 和 counts
    elements = (lines[5].split()) #get a list of elements from the 6th line of the POSCAR file
    counts = [int(value) for value in lines[6].split()]

    # TODO：计算 natoms
    natoms = sum(int(count) for count in counts)

    # TODO：解析 coordinate mode
    coordinate_mode = lines[7].strip()

    # TODO：计算体积
    volume_A3 = np.abs(np.linalg.det(lattice))

    # TODO：返回字典
    header = {
    "title": title,
    "scale": scale,
    "lattice": lattice,
    "elements": elements,
    "counts": counts,
    "natoms": natoms,
    "coordinate_mode": coordinate_mode,
    "volume_A3": volume_A3
    }   
    return header



In [41]:
bulk_header = parse_poscar_header(poscar_path)
print(bulk_header)
print(bulk_header["elements"])
print(bulk_header["counts"])
print(bulk_header["natoms"])
print(bulk_header["volume_A3"])

{'title': 'MgO 64-atom initial supercell', 'scale': 1.0, 'lattice': array([[8.424, 0.   , 0.   ],
       [0.   , 8.424, 0.   ],
       [0.   , 0.   , 8.424]]), 'elements': ['Mg', 'O'], 'counts': [32, 32], 'natoms': 64, 'coordinate_mode': 'Direct', 'volume_A3': np.float64(597.7988490240002)}
['Mg', 'O']
[32, 32]
64
597.7988490240002


## 1.2 Compare bulk and defect compositions

In [42]:
# Load defect POSCAR file exp:q+2 Mg_0
defect_poscar_path = (
    data_root
    / "04_final_defects"
    / "Mg_O"
    / "q+2"
    / "POSCAR"
)
print(defect_poscar_path.exists())

defect_header = parse_poscar_header(defect_poscar_path)
print("defect_elements:", defect_header["elements"])
print("defect_counts:", defect_header["counts"])

print("bulk_elements:", bulk_header["elements"])
print("bulk_counts:", bulk_header["counts"])


True
defect_elements: ['Mg', 'O']
defect_counts: [33, 31]
bulk_elements: ['Mg', 'O']
bulk_counts: [32, 32]


In [43]:
delta_Mg = defect_header["counts"][0] - bulk_header["counts"][0]
delta_O = defect_header["counts"][1] - bulk_header["counts"][1]
print("delta_Mg:", delta_Mg)
print("delta_O:", delta_O)

delta_Mg: 1
delta_O: -1


## 1.3 Load defect metadata

In [47]:
metadata_path = (
    data_root
    / "04_final_defects"
    / "Mg_O"
    / "q+2"
    / "metadata.json"
)

print(metadata_path)
print(metadata_path.exists())

/Users/yu/Desktop/defect_dynamic/data/raw/mgo_defect_thermodynamics_dataset/04_final_defects/Mg_O/q+2/metadata.json
True


In [50]:
# Load json
metadata_text = metadata_path.read_text()
metadata = json.loads(metadata_text) #json->python object

print(type(metadata_text))
print(type(metadata))
print(metadata)

<class 'str'>
<class 'dict'>
{'synthetic': True, 'defect': 'Mg_O', 'charge': 2, 'composition': {'Mg': 33, 'O': 31}, 'atom_changes': {'Mg': 1, 'O': -1}, 'selected_distortion': 'bond_expand_1.20', 'site_multiplicity': 32, 'site_density_cm-3': 4.28e+22, 'degeneracy': 2, 'energy_reference': 'E0'}


In [ ]:
# Check the dictionary
metadata.keys()

defect_name = metadata["defect"]
charge = metadata["charge"]
composition = metadata["composition"]
atom_changes = metadata["atom_changes"]
selected_distortion = metadata["selected_distortion"]
Mg_consistent = np.abs(atom_changes["Mg"] - delta_Mg) == 0
O_consistent = np.abs(atom_changes["O"] - delta_O) == 0

print("Defect:", defect_name)
print("Charge:", charge)
print("Composition:", composition)
print("Atom changes:", atom_changes)
print("Selected distortion:", selected_distortion)
print("Mg_consistent:", Mg_consistent)
print("O_consistent:", O_consistent)

Defect: Mg_O
Charge: 2
Composition: {'Mg': 33, 'O': 31}
Atom changes: {'Mg': 1, 'O': -1}
Selected distortion: bond_expand_1.20
Mg_consistent: True
O_consistent: True


## 1.4 Create an object

In [59]:
poscar_information = {
    "bulk_composition": bulk_composition,
    "defect_composition": defect_composition_from_poscar,
    "atom_changes": atom_changes_from_poscar,
    "bulk_natoms": bulk_header["natoms"],
    "defect_natoms": defect_header["natoms"],
    "bulk_volume_A3": bulk_header["volume_A3"],
    "defect_volume_A3": defect_header["volume_A3"],
}

metadata_information = {
    "defect": metadata["defect"],
    "charge": metadata["charge"],
    "composition": metadata["composition"],
    "atom_changes": metadata["atom_changes"],
    "selected_distortion": metadata["selected_distortion"],
    "site_multiplicity": metadata["site_multiplicity"],
    "site_density_cm-3": metadata["site_density_cm-3"],
    "degeneracy": metadata["degeneracy"],
    "energy_reference": metadata["energy_reference"],
}

validation = {
    "composition_matches": (
        defect_composition_from_poscar
        == metadata["composition"]
    ),
    "atom_changes_match": (
        atom_changes_from_poscar
        == metadata["atom_changes"]
    ),
}

structure_summary = {
    "from_poscar": poscar_information,
    "from_metadata": metadata_information,
    "validation": validation,
}

print("Information parsed from POSCAR:")
for key, value in structure_summary["from_poscar"].items():
    print(f"  {key}: {value}")

print("\nInformation read from metadata:")
for key, value in structure_summary["from_metadata"].items():
    print(f"  {key}: {value}")

print("\nValidation:")
for key, value in structure_summary["validation"].items():
    print(f"  {key}: {value}")

Information parsed from POSCAR:
  bulk_composition: {'Mg': 32, 'O': 32}
  defect_composition: {'Mg': 33, 'O': 31}
  atom_changes: {'Mg': 1, 'O': -1}
  bulk_natoms: 64
  defect_natoms: 64
  bulk_volume_A3: 597.7988490240002
  defect_volume_A3: 597.7988490240002

Information read from metadata:
  defect: Mg_O
  charge: 2
  composition: {'Mg': 33, 'O': 31}
  atom_changes: {'Mg': 1, 'O': -1}
  selected_distortion: bond_expand_1.20
  site_multiplicity: 32
  site_density_cm-3: 4.28e+22
  degeneracy: 2
  energy_reference: E0

Validation:
  composition_matches: True
  atom_changes_match: True
